# Imports

In [1]:
import math
import os
import sys

import numpy as np
import torch
from torchvision.transforms import GaussianBlur
from IPython.core.pylabtools import figsize
from mpmath.identification import transforms
import matplotlib.pyplot as plt
from torchvision.transforms.v2 import RandomApply

from utils.checkpoint import load_checkpoint, save_checkpoint, load_model
from utils.data import get_dataloaders, get_img_from_loader
from evaluate import evaluate, evaluate_with_uncertainty

sys.path.append("..")

from models.lenet import Net as LeNet
from config import Config

from utils.corruptions import gaussian_blur, test_on_corruptions, corruptions_uncertainty
from utils.data import get_img_from_loader



In [2]:
torch.manual_seed(42)

config = Config()
device = config.device
T = 10

# Model
mnist_model = LeNet(
    prior_sigma1=math.exp(0),
    prior_sigma2=math.exp(-6),
    prior_pi=0.5,
    num_classes=10,
    rho_init=-4.5
).to(device)

# Optimizer (needed to load checkpoint)
optimizer = torch.optim.Adam(mnist_model.parameters(), lr=config.learning_rate)
config.model_name = 'lenet_mnist_lrp1em04_logprior10_logprior2m6_priorpip5_v1'
load_checkpoint(mnist_model, optimizer, f'{config.checkpoint_path}/{config.model_name}/{config.get_checkpoint_name(190, date="20260129")}', device)

[checkpoint] Loaded from ../checkpoints/lenet_mnist_lrp1em04_logprior10_logprior2m6_priorpip5_v1/lenet_mnist_lrp1em04_logprior10_logprior2m6_priorpip5_v1_epoch_190_20260129.pth, starting at epoch 191


191

# Пример изображений MNIST с гауссовским размытием

In [ ]:
from tqdm import tqdm

kernel_sizes = [1, 3, 5, 7, 9]


## MNIST

In [13]:
from tqdm import tqdm

mnist_accuracies = []
mnist_total_unc = []
mnist_alea_unc = []
mnist_epis_unc = []

loop = tqdm(kernel_sizes, desc=f"Calculating Accuracy and Uncertainties for MNIST dataset with Gaussian Blur")
for kernel_size in loop:
    blurred_test_loader = get_dataloaders(
        data_dir="../data",
        batch_size=config.test_batch_size,
        num_workers=config.num_workers,
        use_cuda=torch.cuda.is_available(),
        extra_transforms=[
            GaussianBlur(kernel_size)
        ]
    )[2]  # Получаем только тестовый загрузчик

    test_accuracy = evaluate(mnist_model, blurred_test_loader, device=config.device)
    _, uncertainties = evaluate_with_uncertainty(mnist_model, blurred_test_loader, device=config.device, mc_samples=T)
    mnist_accuracies.append(test_accuracy)
    mnist_total_unc.append(uncertainties[0].mean().item())
    mnist_alea_unc.append(uncertainties[1].mean().item())
    mnist_epis_unc.append(uncertainties[2].mean().item())

Calculating Accuracy and Uncertainties for MNIST dataset with Gaussian Blur: 100%|██████████| 5/5 [04:39<00:00, 55.84s/it]
